In [4]:
import pandas as pd
from textblob import TextBlob
import yfinance as yf


df_news = pd.read_csv("RedditNews.csv")
df_stock = pd.read_csv("upload_DJIA_table.csv")


df_news["Date"] = pd.to_datetime(df_news["Date"])
df_stock["Date"] = pd.to_datetime(df_stock["Date"])


df_news_grouped = df_news.groupby("Date")["News"].apply(lambda x: " ".join(x)).reset_index()


df = pd.merge(df_news_grouped, df_stock, on="Date")

df["sentiment"] = df["News"].apply(lambda x: TextBlob(x).sentiment.polarity)
df["headline_length"] = df["News"].apply(len)
df["word_count"] = df["News"].apply(lambda x: len(x.split()))

# ======================
# 6. SECOND DATASET (SP500)
# ======================
sp500 = yf.download("^GSPC", start="2008-06-01", end="2016-07-01")


if isinstance(sp500.columns, pd.MultiIndex):
    sp500.columns = sp500.columns.get_level_values(0)

sp500.reset_index(inplace=True)

sp500 = sp500[["Date", "Close"]]
sp500.rename(columns={"Close": "SP500"}, inplace=True)

# ======================
# 7. MERGE SP500
# ======================
df = pd.merge(df, sp500, on="Date", how="inner")

df["Label"] = (df["Close"].shift(-1) > df["Close"]).astype(int)

df = df.dropna()

df.to_csv("final_dataset.csv", index=False)

print("Final dataset created successfully!")
print(df.head())

/tmp/ipykernel_1535/4254694500.py:37: FutureWarning: YF.download() has changed argument auto_adjust default to True
  sp500 = yf.download("^GSPC", start="2008-06-01", end="2016-07-01")
[*********************100%***********************]  1 of 1 completed


Final dataset created successfully!
        Date                                               News          Open  \
0 2008-08-08  b"Georgia 'downs two Russian warplanes' as cou...  11432.089844   
1 2008-08-11  b'Why wont America and Nato help us? If they w...  11729.669922   
2 2008-08-12  b'Remember that adorable 9-year-old who sang a...  11781.700195   
3 2008-08-13  b' U.S. refuses Israel weapons to attack Iran:...  11632.809570   
4 2008-08-14  b'All the experts admit that we should legalis...  11532.070312   

           High           Low         Close     Volume     Adj Close  \
0  11759.959961  11388.040039  11734.320312  212830000  11734.320312   
1  11867.110352  11675.530273  11782.349609  183190000  11782.349609   
2  11782.349609  11601.519531  11642.469727  173590000  11642.469727   
3  11633.780273  11453.339844  11532.959961  182550000  11532.959961   
4  11718.280273  11450.889648  11615.929688  159790000  11615.929688   

   sentiment  headline_length  word_count   